# Phase 2: Evolutionary Algorithm Optimization — `EA_Optimization.ipynb`

**Team 1 — NYC Taxi Trip Duration**

---


In [1]:
# 🔍 GPU Preflight Check (Run this BEFORE the EA cell)
import os
import torch

print("=== GPU PRECHECK ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print("Current device:", torch.cuda.current_device())
else:
    print("Running on CPU — STOP if you expected GPU")

print(
    "NRP image hint:",
    os.environ.get("JUPYTER_IMAGE_SPEC")
    or os.environ.get("CONTAINER_IMAGE")
    or os.environ.get("JUPYTERHUB_IMAGE")
    or os.environ.get("IMAGE_NAME")
    or "UNKNOWN",
)
print("=== END PRECHECK ===")


=== GPU PRECHECK ===
CUDA available: False
GPU count: 0
Running on CPU — STOP if you expected GPU
NRP image hint: UNKNOWN
=== END PRECHECK ===


## Cell 1 — Imports, Constants & Device

Identical configuration to Phase 1 `NN_Analysis.ipynb` Cell 1.
## EA Data & Evaluation Protocol

Data Splits:
- Train set: Used ONLY for neural network training.
- Validation set: Used ONLY for EA fitness evaluation.
- Test set: Completely untouched during EA evolution.

Split Policy:
- Phase 1 official 70/15/15 split reused.
- Same random seed (SEED = 42).
- No reshuffling or resplitting inside this notebook.

Evaluation Discipline:
- EA fitness is computed using Validation R² on the original scale, with Validation MAPE tracked as a secondary metric.
- Test metrics are computed exactly once after EA completes.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Constants (identical to Phase 1) ──
SEED = 42
NROWS = 1_000_000
TARGET = "trip_duration"
CLIP_MIN = -2.0
CLIP_MAX = 13.0

DATA_PATH = Path("../data/train.csv")
DATA_URL = "https://github.com/DrAlzahraniProjects/csusb_spring26_cse5140_team1/releases/download/v1.0/train.csv"

ARTIFACTS_DIR = Path("../artifacts/phase2")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("EA_Optimization.ipynb — Phase 2")
print("=" * 60)
print(f"  SEED:      {SEED}")
print(f"  NROWS:     {NROWS:,}")
print(f"  TARGET:    {TARGET}")
print(f"  DATA_PATH: {DATA_PATH}")
print(f"  Device:    {device}")
print("=" * 60)


EA_Optimization.ipynb — Phase 2
  SEED:      42
  NROWS:     1,000,000
  TARGET:    trip_duration
  DATA_PATH: ../data/train.csv
  Device:    cpu


## Cell 2 — Deterministic Seeding

Same `seed_everything()` from Phase 1 — locks all RNGs.


In [3]:
import random

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"All RNGs seeded with {SEED}")


All RNGs seeded with 42


---
# Step 1.1 — Data Verification

Verify that Phase 2 uses the exact same dataset, shuffle, and splits as Phase 1.

## Cell 3 — Load Dataset

Load first 1M rows and shuffle with `SEED=42`, identical to Phase 1.


In [4]:
if not DATA_PATH.exists():
    import urllib.request
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")
else:
    print("Dataset already exists at", DATA_PATH)

df = pd.read_csv(DATA_PATH, nrows=NROWS)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Shuffled with seed={SEED}")
print(f"First 3 IDs after shuffle: {df['id'].head(3).tolist()}")


Download complete.
Loaded: 1,000,000 rows, 11 columns
Shuffled with seed=42
First 3 IDs after shuffle: ['id3435429', 'id2267606', 'id3771460']


## Cell 4 — Train / Validation / Test Splits

Phase 2 reuses the official Phase 1 split:
- 70% Train
- 15% Validation
- 15% Test
Seed = 42

No re-shuffling or re-splitting occurs later in this notebook.


In [5]:
# ── Official 70/15/15 split (reuse Phase 1 policy) ──
dev_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED)
train_df, val_df = train_test_split(dev_df, test_size=(0.15/0.85), random_state=SEED)

print("Split sizes:")
print(f"  train_df: {len(train_df):>8,} rows  (70%)")
print(f"  val_df:   {len(val_df):>8,} rows  (15%)")
print(f"  test_df:  {len(test_df):>8,} rows  (15%)")
print(f"  total:    {len(train_df) + len(val_df) + len(test_df):>8,} rows")

# ── Assertions ──
assert len(train_df) + len(val_df) + len(test_df) == NROWS, "Row count mismatch!"
assert abs(len(train_df) - 700_000) < 1000, f"Train expected ~700k, got {len(train_df)}"
assert abs(len(val_df) - 150_000) < 1000,   f"Val expected ~150k, got {len(val_df)}"
assert abs(len(test_df) - 150_000) < 1000,  f"Test expected ~150k, got {len(test_df)}"

print("\n✅ All split assertions passed — 70/15/15 verified")

Split sizes:
  train_df:  700,000 rows  (70%)
  val_df:    150,000 rows  (15%)
  test_df:   150,000 rows  (15%)
  total:    1,000,000 rows

✅ All split assertions passed — 70/15/15 verified


## Cell 6 — Step 1.1 Summary

Data verification is complete. The table below confirms consistency with Phase 1.


In [6]:
verification_summary = pd.DataFrame([
    {"Check": "Dataset rows loaded", "Expected": "1,000,000", "Actual": f"{len(df):,}", "Status": "✅"},
    {"Check": "Random seed", "Expected": "42", "Actual": str(SEED), "Status": "✅"},

    {"Check": "Train rows", "Expected": "~700,000", "Actual": f"{len(train_df):,}", "Status": "✅"},
    {"Check": "Validation rows", "Expected": "~150,000", "Actual": f"{len(val_df):,}", "Status": "✅"},
    {"Check": "Test rows", "Expected": "~150,000", "Actual": f"{len(test_df):,}", "Status": "✅"},

    {"Check": "Total rows", "Expected": "1,000,000",
     "Actual": f"{len(train_df)+len(val_df)+len(test_df):,}", "Status": "✅"},

    {"Check": "No re-splitting", "Expected": "Single split only",
     "Actual": f"random_state={SEED}", "Status": "✅"},
])

print("=" * 60)
print("STEP 1.1 — DATA VERIFICATION RESULTS (70/15/15)")
print("=" * 60)
print(verification_summary.to_string(index=False))
print("\n✅ Step 1.1 complete — dataset and 70/15/15 splits verified")

STEP 1.1 — DATA VERIFICATION RESULTS (70/15/15)
              Check          Expected          Actual Status
Dataset rows loaded         1,000,000       1,000,000      ✅
        Random seed                42              42      ✅
         Train rows          ~700,000         700,000      ✅
    Validation rows          ~150,000         150,000      ✅
          Test rows          ~150,000         150,000      ✅
         Total rows         1,000,000       1,000,000      ✅
    No re-splitting Single split only random_state=42      ✅

✅ Step 1.1 complete — dataset and 70/15/15 splits verified


---
# Step 1.2 — Feature & Preprocessing Consistency

Verify that the same feature engineering pipeline and scaling procedure from Phase 1 are applied identically.

## Cell 7 — Feature Engineering Functions

Same `haversine_km()` and `build_features()` copied from Phase 1.
Feature families:
- **Temporal:** pickup_hour, pickup_dow, pickup_month + cyclical (sin/cos)
- **Spatial:** delta_lat, delta_lon, haversine_km
- **Proxies:** passenger_count, store_and_fwd_Y, vendor one-hot


In [7]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in km."""
    R = 6371.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def build_features(dfin: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=dfin.index)

    # Temporal
    dt = pd.to_datetime(dfin["pickup_datetime"], errors="coerce")
    X["pickup_hour"]  = dt.dt.hour.fillna(0).astype(int)
    X["pickup_dow"]   = dt.dt.dayofweek.fillna(0).astype(int)
    X["pickup_month"] = dt.dt.month.fillna(0).astype(int)

    X["hour_sin"] = np.sin(2*np.pi*X["pickup_hour"]/24)
    X["hour_cos"] = np.cos(2*np.pi*X["pickup_hour"]/24)
    X["dow_sin"]  = np.sin(2*np.pi*X["pickup_dow"]/7)
    X["dow_cos"]  = np.cos(2*np.pi*X["pickup_dow"]/7)

    # Spatial
    X["delta_lat"] = (dfin["dropoff_latitude"] - dfin["pickup_latitude"]).astype(float)
    X["delta_lon"] = (dfin["dropoff_longitude"] - dfin["pickup_longitude"]).astype(float)
    X["haversine_km"] = haversine_km(
        dfin["pickup_latitude"].astype(float),
        dfin["pickup_longitude"].astype(float),
        dfin["dropoff_latitude"].astype(float),
        dfin["dropoff_longitude"].astype(float),
    )

    # Proxies
    X["passenger_count"] = pd.to_numeric(dfin["passenger_count"], errors="coerce").fillna(0.0)
    X["store_and_fwd_Y"] = (dfin["store_and_fwd_flag"].astype(str).str.upper() == "Y").astype(int)

    vendor_oh = pd.get_dummies(dfin["vendor_id"].astype(str), prefix="vendor", drop_first=False)
    X = pd.concat([X, vendor_oh], axis=1)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X

print("Feature functions defined (identical to Phase 1)")


Feature functions defined (identical to Phase 1)


## Cell 8 — Build Feature Matrices & Align Columns

Apply `build_features()` to each split. Align val/holdout columns to train (critical for one-hot dummies).


In [8]:
# Build train/val/test features using the SAME Phase 1 pipeline
X_train = build_features(train_df)
feature_cols = X_train.columns

X_val  = build_features(val_df).reindex(columns=feature_cols, fill_value=0.0)
X_test = build_features(test_df).reindex(columns=feature_cols, fill_value=0.0)

# --- Keep BOTH original-scale and log-scale targets ---
# Original scale = required for EA primary fitness (validation R²) and secondary MAPE
y_train_orig = train_df[TARGET].to_numpy().astype(np.float64)
y_val_orig   = val_df[TARGET].to_numpy().astype(np.float64)
y_test_orig  = test_df[TARGET].to_numpy().astype(np.float64)

# Log scale = optional supplemental diagnostics / continuity with Phase 1
y_train_log = np.log1p(y_train_orig)
y_val_log   = np.log1p(y_val_orig)
y_test_log  = np.log1p(y_test_orig)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Targets prepared:")
print("  Original scale:", y_train_orig.shape, y_val_orig.shape, y_test_orig.shape)
print("  Log scale:     ", y_train_log.shape, y_val_log.shape, y_test_log.shape)


X_train: (700000, 14) X_val: (150000, 14) X_test: (150000, 14)
Targets prepared:
  Original scale: (700000,) (150000,) (150000,)
  Log scale:      (700000,) (150000,) (150000,)


## Cell 9 — Feature Family Verification

Assert all expected features from Phase 1 are present in the exact same order.


In [9]:
EXPECTED_FEATURES = {
    "temporal": ["pickup_hour", "pickup_dow", "pickup_month",
                 "hour_sin", "hour_cos", "dow_sin", "dow_cos"],
    "spatial":  ["delta_lat", "delta_lon", "haversine_km"],
    "proxies":  ["passenger_count", "store_and_fwd_Y"],
}

print("Feature family verification:")
all_expected = []
for family, cols in EXPECTED_FEATURES.items():
    missing = [c for c in cols if c not in feature_cols]
    if missing:
        print(f"  ❌ {family}: MISSING {missing}")
    else:
        print(f"  ✅ {family}: {cols}")
    all_expected.extend(cols)

# Check for vendor one-hot columns
vendor_cols = [c for c in feature_cols if c.startswith("vendor_")]
assert len(vendor_cols) >= 1, "No vendor one-hot columns found!"
print(f"  ✅ vendor one-hot: {vendor_cols}")

# Verify no NaN or Inf values
assert not X_train.isnull().any().any(), "X_train has NaN values!"
assert not X_val.isnull().any().any(), "X_val has NaN values!"
assert not X_test.isnull().any().any(), "X_test has NaN values!"
print(f"\n  ✅ No NaN or Inf in any feature matrix")

# Verify train/val/holdout have same number of columns
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1], "Column count mismatch!"
print(f"  ✅ All splits have {X_train.shape[1]} features")


Feature family verification:
  ✅ temporal: ['pickup_hour', 'pickup_dow', 'pickup_month', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
  ✅ spatial: ['delta_lat', 'delta_lon', 'haversine_km']
  ✅ proxies: ['passenger_count', 'store_and_fwd_Y']
  ✅ vendor one-hot: ['vendor_1', 'vendor_2']

  ✅ No NaN or Inf in any feature matrix
  ✅ All splits have 14 features


## Cell 10 — Z-Score Standardization

Scale using training set statistics only (same as Phase 1). Save mu/sigma for reproducibility.


In [10]:
mu    = X_train.mean()
sigma = X_train.std().replace(0, 1)

X_train_s   = (X_train   - mu) / sigma
X_val_s     = (X_val     - mu) / sigma
X_test_s    = (X_test - mu) / sigma
X_holdout_s = X_test_s  # backward-compatible alias

# Save scaling artifacts
mu.to_csv(ARTIFACTS_DIR / "mu.csv")
sigma.to_csv(ARTIFACTS_DIR / "sigma.csv")
pd.Series(feature_cols, name="feature").to_csv(ARTIFACTS_DIR / "feature_cols.csv", index=False)

print("Z-score scaling applied (train stats only):")
print(f"  X_train_s mean (should be ~0): {X_train_s.mean().mean():.6f}")
print(f"  X_train_s std  (should be ~1): {X_train_s.std().mean():.6f}")
print(f"\nSaved mu, sigma, feature_cols to {ARTIFACTS_DIR}")


Z-score scaling applied (train stats only):
  X_train_s mean (should be ~0): 0.000000
  X_train_s std  (should be ~1): 1.000000

Saved mu, sigma, feature_cols to ../artifacts/phase2


## Cell 11 — Scaling Verification

Cross-check that scaled feature statistics are reasonable and consistent.


In [11]:
print("Scaled feature statistics (train):")
print(X_train_s.describe().round(4).to_string())

print("\n\nScaled feature statistics (validation):")
print(X_val_s.describe().round(4).to_string())

# The val set should have means close to 0 and stds close to 1
# (not exact, because scaling uses train stats)
val_means = X_val_s.mean()
print(f"\nVal scaled means range: [{val_means.min():.4f}, {val_means.max():.4f}]")
print("(Should be close to 0 — minor drift is expected since scaling uses train stats)")


Scaled feature statistics (train):
       pickup_hour   pickup_dow  pickup_month     hour_sin     hour_cos      dow_sin      dow_cos    delta_lat    delta_lon  haversine_km  passenger_count  store_and_fwd_Y     vendor_1     vendor_2
count  700000.0000  700000.0000   700000.0000  700000.0000  700000.0000  700000.0000  700000.0000  700000.0000  700000.0000   700000.0000      700000.0000      700000.0000  700000.0000  700000.0000
mean        0.0000       0.0000        0.0000       0.0000      -0.0000       0.0000       0.0000      -0.0000       0.0000       -0.0000          -0.0000          -0.0000       0.0000       0.0000
std         1.0000       1.0000        1.0000       1.0000       1.0000       1.0000       1.0000       1.0000       1.0000        1.0000           1.0000           1.0000       1.0000       1.0000
min        -2.1286      -1.5612       -1.4974      -1.1862      -1.2943      -1.3659      -1.2253    -304.4937    -161.1502       -0.7646          -1.2666          -0.0748  

## Cell 12 — Step 1.2 Summary

Feature and preprocessing consistency is verified.


In [12]:
feature_summary = pd.DataFrame([
    {"Check": "Feature count", "Expected": "Phase 1 count", "Actual": str(len(feature_cols)), "Status": "✅"},
    {"Check": "Temporal features", "Expected": "7", "Actual": str(sum(1 for c in feature_cols if c in EXPECTED_FEATURES["temporal"])), "Status": "✅"},
    {"Check": "Spatial features", "Expected": "3", "Actual": str(sum(1 for c in feature_cols if c in EXPECTED_FEATURES["spatial"])), "Status": "✅"},
    {"Check": "Proxy features", "Expected": "2+", "Actual": str(sum(1 for c in feature_cols if c in EXPECTED_FEATURES["proxies"]) + len(vendor_cols)), "Status": "✅"},
    {"Check": "No NaN/Inf", "Expected": "True", "Actual": "True", "Status": "✅"},
    {"Check": "Column alignment", "Expected": "All splits equal", "Actual": f"All = {X_train.shape[1]} cols", "Status": "✅"},
    {"Check": "Scaling method", "Expected": "Z-score (train only)", "Actual": "Z-score (train only)", "Status": "✅"},
    {"Check": "Scaling artifacts saved", "Expected": "mu.csv, sigma.csv", "Actual": "Saved", "Status": "✅"},
])

print("=" * 60)
print("STEP 1.2 — FEATURE & PREPROCESSING CONSISTENCY RESULTS")
print("=" * 60)
display(feature_summary.to_string(index=False))
print("\n✅ Step 1.2 complete — feature pipeline and scaling verified identical to Phase 1")


STEP 1.2 — FEATURE & PREPROCESSING CONSISTENCY RESULTS


'                  Check             Expected               Actual Status\n          Feature count        Phase 1 count                   14      ✅\n      Temporal features                    7                    7      ✅\n       Spatial features                    3                    3      ✅\n         Proxy features                   2+                    4      ✅\n             No NaN/Inf                 True                 True      ✅\n       Column alignment     All splits equal        All = 14 cols      ✅\n         Scaling method Z-score (train only) Z-score (train only)      ✅\nScaling artifacts saved    mu.csv, sigma.csv                Saved      ✅'


✅ Step 1.2 complete — feature pipeline and scaling verified identical to Phase 1


## Steps 1.1 & 1.2 Complete — Handoff Summary

The data foundation for Phase 2 is verified and ready. The following variables are available for downstream cells:

| Variable | Description |
|---|---|
| `train_df`, `val_df`, `test_df` | Raw DataFrames (same split policy as Phase 1) |
| `X_train_s`, `X_val_s`, `X_test_s` | Scaled feature matrices |
| `X_holdout_s` | Backward-compatible alias for `X_test_s` |
| `y_train_orig`, `y_val_orig`, `y_test_orig` | Original-scale target arrays (seconds) |
| `y_train_log`, `y_val_log`, `y_test_log` | Log-transformed targets |
| `feature_cols` | Ordered list of feature column names |
| `mu`, `sigma` | Training scaling parameters |
| `SEED`, `CLIP_MIN`, `CLIP_MAX`, `device` | Global constants |

Next: Step 1.3 (Reference Model Confirmation) → Step 2 (EA Optimization) → Step 3 (Evaluation)


In [13]:
print("=" * 60)
print("STEPS 1.1 & 1.2 — HANDOFF SUMMARY")
print("=" * 60)
print(f"  Seed:             {SEED}")
print(f"  Rows:             {NROWS:,}")
print(f"  Train:            {len(train_df):,} rows  |  X_train_s: {X_train_s.shape}")
print(f"  Validation:       {len(val_df):,} rows  |  X_val_s:   {X_val_s.shape}")
print(f"  Test:          {len(test_df):,} rows  |  X_test_s: {X_test_s.shape}")
print(f"  Features:         {len(feature_cols)}")
print(f"  Scaling:          Z-score (train mu/sigma)")
print(f"  Device:           {device}")
print(f"  Artifacts saved:  {ARTIFACTS_DIR}")
print("=" * 60)
print("\n✅ Ready for Step 1.3 (Reference Model Confirmation)")


STEPS 1.1 & 1.2 — HANDOFF SUMMARY
  Seed:             42
  Rows:             1,000,000
  Train:            700,000 rows  |  X_train_s: (700000, 14)
  Validation:       150,000 rows  |  X_val_s:   (150000, 14)
  Test:          150,000 rows  |  X_test_s: (150000, 14)
  Features:         14
  Scaling:          Z-score (train mu/sigma)
  Device:           cpu
  Artifacts saved:  ../artifacts/phase2

✅ Ready for Step 1.3 (Reference Model Confirmation)


---
# Step 1.3 — GA Compute Budget & Convergence Logging

This section locks the **Genetic Algorithm (GA)** execution budget and defines a minimal logging structure for convergence plots.

**Hard requirement:** the compute budget must be fixed (population × generations) and printed during execution.


## Fixed Compute Budget

- Population: **20**
- Generations: **15**
- Total Evaluations: **600**

This budget is fixed and must not change during execution.


In [14]:
# ── GA Hyperparameter Optimization Budget (FIXED) ──
# IMPORTANT: Do not modify these values during a run (spec compliance: fixed compute budget).

POPULATION_SIZE = 20
GENERATIONS = 15
MUTATION_RATE = 0.10
CROSSOVER_RATE = 0.80
ELITE_COUNT = 2

TOTAL_EVALUATIONS = POPULATION_SIZE * GENERATIONS
print("GA Compute Budget")
print("  Population   =", POPULATION_SIZE)
print("  Generations  =", GENERATIONS)
print("  Evaluations  =", TOTAL_EVALUATIONS)


GA Compute Budget
  Population   = 20
  Generations  = 15
  Evaluations  = 300


In [15]:
# Step 2 uses the fitness/evaluation implementation defined in the main EA cell below.
# We intentionally do NOT define a separate executable `fitness(individual)` here,
# because duplicate fitness definitions make the notebook harder to audit and can
# cause accidental inconsistency in the optimization objective.

print("Cell 28 note: standalone fitness stub removed.")
print("The authoritative EA fitness implementation appears in the main EA optimization cell.")
print("Primary objective there will be validation R² on original scale; MAPE is tracked as secondary.")


Cell 28 note: standalone fitness stub removed.
The authoritative EA fitness implementation appears in the main EA optimization cell.
Primary objective there will be validation R² on original scale; MAPE is tracked as secondary.


In [16]:
# -----------------------------
# Convergence logging utilities
# -----------------------------
generation_logs = []

def log_generation_stats(generation, population_metrics):
    """
    population_metrics: list of dicts with keys:
        - val_r2
        - val_mape
    """
    val_r2s = [m["val_r2"] for m in population_metrics]
    val_mapes = [m["val_mape"] for m in population_metrics]

    row = {
        "generation": generation,
        "best_val_r2": float(np.max(val_r2s)),
        "mean_val_r2": float(np.mean(val_r2s)),
        "best_val_mape": float(np.min(val_mapes)),   # lower is better
        "mean_val_mape": float(np.mean(val_mapes)),
    }
    generation_logs.append(row)
    return row

print("Initialized generation logging with both validation R² and validation MAPE.")


Initialized generation logging with both validation R² and validation MAPE.


In [17]:
# ============================================================
# Phase 2 — EA Optimization (GPU-accelerated, compliant version)
# Primary EA fitness: validation R² on ORIGINAL scale
# Secondary tracked metric: validation MAPE on ORIGINAL scale
# Training target: log1p(duration) for stability (fitness/reporting
#                  are still on original scale)
# ============================================================

import os
import sys
import time
import json
import copy
import random
import platform
import subprocess
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# 0) Sanity checks: required objects from earlier cells
# ------------------------------------------------------------
required_names = [
    "SEED", "ARTIFACTS_DIR", "device",
    "X_train_s", "X_val_s", "X_test_s",
    "y_train_orig", "y_val_orig", "y_test_orig",
    "y_train_log", "y_val_log", "y_test_log",
    "log_generation_stats"
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f"Missing required variables from earlier cells: {missing}")

ARTIFACTS_DIR = Path(ARTIFACTS_DIR)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = ARTIFACTS_DIR / "plots"
LOG_DIR = ARTIFACTS_DIR / "ea_trial_logs"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Artifacts:", ARTIFACTS_DIR.resolve())

# ------------------------------------------------------------
# 1) NRP / environment snapshot (auditable evidence)
# ------------------------------------------------------------
def _safe_cmd(cmd):
    try:
        return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)
    except Exception as e:
        return f"(command failed: {cmd} -> {e})\n"

image_hint = (
    os.environ.get("JUPYTER_IMAGE_SPEC")
    or os.environ.get("CONTAINER_IMAGE")
    or os.environ.get("JUPYTERHUB_IMAGE")
    or os.environ.get("IMAGE_NAME")
    or "UNKNOWN"
)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
env_snapshot = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "platform": platform.platform(),
    "python_version": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "torch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "device": str(device),
    "gpu_count": int(gpu_count),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "nrp_image_hint": image_hint,
}

env_path = ARTIFACTS_DIR / "environment_snapshot_phase2.json"
env_path.write_text(json.dumps(env_snapshot, indent=2), encoding="utf-8")

print("Saved environment snapshot to:", env_path.resolve())
print("NRP image hint:", image_hint)

# ------------------------------------------------------------
# 2) GPU acceleration controls
# ------------------------------------------------------------
USE_CUDA = (torch.cuda.is_available() and str(device).startswith("cuda"))
NUM_GPUS = torch.cuda.device_count() if USE_CUDA else 0
USE_DATA_PARALLEL = NUM_GPUS > 1
USE_AMP = USE_CUDA
PIN_MEMORY = USE_CUDA
CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = min(4, max(0, CPU_COUNT - 1))
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None
EVAL_BATCH_SIZE = 65536 if USE_CUDA else 16384

# Keep seeds fixed for reproducibility.
# We leave cudnn.benchmark=False to avoid changing execution paths run-to-run.
if USE_CUDA:
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("\nAcceleration config:")
print("  USE_CUDA          :", USE_CUDA)
print("  NUM_GPUS          :", NUM_GPUS)
print("  USE_DATA_PARALLEL :", USE_DATA_PARALLEL)
print("  USE_AMP           :", USE_AMP)
print("  NUM_WORKERS       :", NUM_WORKERS)
print("  PIN_MEMORY        :", PIN_MEMORY)
print("  EVAL_BATCH_SIZE   :", EVAL_BATCH_SIZE)

# ------------------------------------------------------------
# 3) Reproducibility helpers
# ------------------------------------------------------------
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)

# ------------------------------------------------------------
# 4) Metrics + target handling
# ------------------------------------------------------------
TRAIN_IN_LOG_SPACE = True
CLIP_MIN_LOG = -2.0
CLIP_MAX_LOG = 13.0

def safe_expm1(yhat_log, clip_min=CLIP_MIN_LOG, clip_max=CLIP_MAX_LOG):
    yhat_log = np.asarray(yhat_log).reshape(-1)
    yhat_log = np.clip(yhat_log, clip_min, clip_max)
    return np.expm1(yhat_log)

def mape(y_true, y_pred, eps=1.0):
    y_true = np.asarray(y_true).reshape(-1).astype(np.float64)
    y_pred = np.asarray(y_pred).reshape(-1).astype(np.float64)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def regression_metrics_original(y_true_orig, y_pred_orig):
    return {
        "r2": float(r2_score(y_true_orig, y_pred_orig)),
        "rmse": float(mean_squared_error(y_true_orig, y_pred_orig, squared=False)),
        "mae": float(mean_absolute_error(y_true_orig, y_pred_orig)),
        "mape": float(mape(y_true_orig, y_pred_orig)),
    }

def regression_metrics_log(y_true_log, y_pred_log):
    return {
        "r2_log": float(r2_score(y_true_log, y_pred_log)),
        "rmse_log": float(mean_squared_error(y_true_log, y_pred_log, squared=False)),
        "mae_log": float(mean_absolute_error(y_true_log, y_pred_log)),
    }

def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

# ------------------------------------------------------------
# 5) Data interface
# ------------------------------------------------------------
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(np.asarray(X), dtype=torch.float32)
        self.y = torch.as_tensor(np.asarray(y), dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]

X_train_np = np.asarray(X_train_s, dtype=np.float32)
X_val_np   = np.asarray(X_val_s, dtype=np.float32)
X_test_np  = np.asarray(X_test_s, dtype=np.float32)

# Training target can be log-space, but fitness/reporting remain original-scale.
y_train_fit = y_train_log if TRAIN_IN_LOG_SPACE else y_train_orig
y_val_fit   = y_val_log if TRAIN_IN_LOG_SPACE else y_val_orig
y_test_fit  = y_test_log if TRAIN_IN_LOG_SPACE else y_test_orig

# Build shared datasets/tensors once to reduce repeated setup cost.
train_ds = TabularDataset(X_train_np, y_train_fit)
X_val_tensor = torch.as_tensor(X_val_np, dtype=torch.float32, device=device)
X_test_tensor = torch.as_tensor(X_test_np, dtype=torch.float32, device=device)

def make_train_loader(batch_size, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    kwargs = dict(
        dataset=train_ds,
        batch_size=int(batch_size),
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        generator=generator,
    )
    if PREFETCH_FACTOR is not None:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

# ------------------------------------------------------------
# 6) Model / individual encoding
# ------------------------------------------------------------
SEARCH_SPACE = {
    "learning_rate": (1e-4, 1e-2),                   # continuous (log-uniform)
    "dropout": (0.05, 0.30),                         # continuous
    "layers": [(128, 64), (256, 128), (512, 256)],  # discrete
    "weight_decay": (1e-6, 1e-3),                   # continuous (log-uniform)
    "batch_size": [1024, 2048, 4096] if USE_CUDA else [1024, 2048],
}

POPULATION_SIZE = 20
GENERATIONS = 15
MUTATION_RATE = 0.20
CROSSOVER_RATE = 0.80
ELITE_COUNT = 2

# Fixed internal training budget per candidate
EPOCHS_PER_CANDIDATE = 3
PATIENCE = 3

print("\nEA compute budget:")
print("  Population size :", POPULATION_SIZE)
print("  Generations     :", GENERATIONS)
print("  Theoretical eval:", POPULATION_SIZE * GENERATIONS)
print("  Mutation rate   :", MUTATION_RATE)
print("  Crossover rate  :", CROSSOVER_RATE)
print("  Elite count     :", ELITE_COUNT)

def sample_log_uniform(low, high, rng):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def create_individual(rng=None):
    rng = np.random.default_rng(SEED) if rng is None else rng
    return {
        "learning_rate": sample_log_uniform(*SEARCH_SPACE["learning_rate"], rng=rng),
        "dropout": float(rng.uniform(*SEARCH_SPACE["dropout"])),
        "layers": tuple(SEARCH_SPACE["layers"][int(rng.integers(0, len(SEARCH_SPACE["layers"])))]),
        "weight_decay": sample_log_uniform(*SEARCH_SPACE["weight_decay"], rng=rng),
        "batch_size": int(SEARCH_SPACE["batch_size"][int(rng.integers(0, len(SEARCH_SPACE["batch_size"])))]),
    }

def individual_key(ind):
    return (
        round(float(ind["learning_rate"]), 10),
        round(float(ind["dropout"]), 10),
        tuple(ind["layers"]),
        round(float(ind["weight_decay"]), 10),
        int(ind["batch_size"]),
    )

def unwrap_model(model):
    return model.module if hasattr(model, "module") else model

def build_model_from_individual(individual, in_dim):
    layers = individual["layers"]
    dropout = float(individual["dropout"])
    net = []
    prev = in_dim
    for h in layers:
        net.append(nn.Linear(prev, h))
        net.append(nn.ReLU())
        net.append(nn.Dropout(dropout))
        prev = h
    net.append(nn.Linear(prev, 1))
    base_model = nn.Sequential(*net).to(device)

    if USE_DATA_PARALLEL:
        base_model = nn.DataParallel(base_model)
    return base_model

# ------------------------------------------------------------
# 7) Prediction helpers (batched on-device inference)
# ------------------------------------------------------------
def model_predict_fit_space(model, X_tensor):
    model.eval()
    preds = []
    with torch.inference_mode():
        with autocast_context():
            for start in range(0, X_tensor.shape[0], EVAL_BATCH_SIZE):
                xb = X_tensor[start:start + EVAL_BATCH_SIZE]
                out = model(xb).view(-1)
                preds.append(out.detach().float().cpu())
    preds = torch.cat(preds).numpy().reshape(-1)
    if TRAIN_IN_LOG_SPACE:
        preds = np.clip(preds, CLIP_MIN_LOG, CLIP_MAX_LOG)
    return preds

def model_predict_original(model, X_tensor):
    pred_fit = model_predict_fit_space(model, X_tensor)
    if TRAIN_IN_LOG_SPACE:
        return safe_expm1(pred_fit)
    return np.maximum(pred_fit, 0.0)

# ------------------------------------------------------------
# 8) Train/evaluate one candidate
# ------------------------------------------------------------
def evaluate_candidate(individual, trial_seed):
    """
    Train on TRAIN only.
    Select epoch by VALIDATION R² on ORIGINAL scale.
    Return both primary metric (val_r2) and secondary metric (val_mape).
    """
    seed_everything(trial_seed)

    model = build_model_from_individual(individual, in_dim=X_train_np.shape[1])
    train_loader = make_train_loader(batch_size=individual["batch_size"], seed=trial_seed)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(individual["learning_rate"]),
        weight_decay=float(individual["weight_decay"]),
    )
    loss_fn = nn.SmoothL1Loss(beta=0.5) if TRAIN_IN_LOG_SPACE else nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_state = None
    best_epoch = 0
    best_val_r2 = -np.inf
    best_epoch_metrics = None
    patience_counter = 0
    history = []

    for epoch in range(1, EPOCHS_PER_CANDIDATE + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=PIN_MEMORY)
            yb = yb.to(device, non_blocking=PIN_MEMORY).view(-1)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                preds = model(xb).view(-1)
                if TRAIN_IN_LOG_SPACE:
                    preds = torch.clamp(preds, CLIP_MIN_LOG, CLIP_MAX_LOG)
                loss = loss_fn(preds, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.detach().item()) * xb.size(0)
            seen += xb.size(0)

        train_loss = running_loss / max(seen, 1)

        # Validation metrics on ORIGINAL scale (required for EA fitness)
        val_pred_orig = model_predict_original(model, X_val_tensor)
        val_metrics_orig = regression_metrics_original(y_val_orig, val_pred_orig)

        # Optional supplemental diagnostics in log space
        val_pred_fit = model_predict_fit_space(model, X_val_tensor)
        val_metrics_log = regression_metrics_log(y_val_fit, val_pred_fit)

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            **{f"val_{k}": v for k, v in val_metrics_orig.items()},
            **{f"val_{k}": v for k, v in val_metrics_log.items()},
        }
        history.append(row)

        # Early stopping within candidate training: based on PRIMARY metric (val_r2)
        if val_metrics_orig["r2"] > best_val_r2 + 1e-6:
            best_val_r2 = val_metrics_orig["r2"]
            best_epoch = epoch
            best_epoch_metrics = {
                **val_metrics_orig,
                **val_metrics_log,
            }
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in unwrap_model(model).state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    if best_state is None:
        raise RuntimeError("No valid state captured during candidate evaluation.")
    unwrap_model(model).load_state_dict(best_state)

    hist_path = LOG_DIR / f"candidate_{abs(hash(individual_key(individual))) % 10**10}_{trial_seed}.csv"
    pd.DataFrame(history).to_csv(hist_path, index=False)

    result = {
        "model": model,
        "best_epoch": int(best_epoch),
        "history_csv": str(hist_path),
        "val_r2": float(best_epoch_metrics["r2"]),
        "val_rmse": float(best_epoch_metrics["rmse"]),
        "val_mae": float(best_epoch_metrics["mae"]),
        "val_mape": float(best_epoch_metrics["mape"]),
        "val_r2_log": float(best_epoch_metrics["r2_log"]),
        "val_rmse_log": float(best_epoch_metrics["rmse_log"]),
        "val_mae_log": float(best_epoch_metrics["mae_log"]),
    }
    return result

# ------------------------------------------------------------
# 9) GA operators
# ------------------------------------------------------------
def initialize_population(size, seed=SEED):
    rng = np.random.default_rng(seed)
    return [create_individual(rng) for _ in range(size)]

def tournament_selection(population, metrics_list, k=3, seed=None):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(population), size=k, replace=False)
    best_idx = max(idxs, key=lambda i: metrics_list[i]["val_r2"])  # maximize validation R²
    return copy.deepcopy(population[best_idx])

def crossover(parent1, parent2, seed=None):
    rng = np.random.default_rng(seed)
    child = {}
    for key in parent1.keys():
        child[key] = copy.deepcopy(parent1[key] if rng.random() < 0.5 else parent2[key])
    return child

def mutate(individual, mutation_rate=MUTATION_RATE, seed=None):
    rng = np.random.default_rng(seed)
    child = copy.deepcopy(individual)

    if rng.random() < mutation_rate:
        child["learning_rate"] = sample_log_uniform(*SEARCH_SPACE["learning_rate"], rng=rng)

    if rng.random() < mutation_rate:
        child["dropout"] = float(rng.uniform(*SEARCH_SPACE["dropout"]))

    if rng.random() < mutation_rate:
        child["layers"] = tuple(SEARCH_SPACE["layers"][int(rng.integers(0, len(SEARCH_SPACE["layers"])))])

    if rng.random() < mutation_rate:
        child["weight_decay"] = sample_log_uniform(*SEARCH_SPACE["weight_decay"], rng=rng)

    if rng.random() < mutation_rate:
        child["batch_size"] = int(SEARCH_SPACE["batch_size"][int(rng.integers(0, len(SEARCH_SPACE["batch_size"])))])

    return child

# ------------------------------------------------------------
# 10) Fitness cache + fitness function
# ------------------------------------------------------------
fitness_cache = {}
fitness_calls = 0

def fitness(individual, eval_id):
    """
    Authoritative EA fitness.
    Returns PRIMARY metric only: validation R² on ORIGINAL scale.
    Secondary metric (validation MAPE) is also computed and cached.
    """
    global fitness_calls

    key = individual_key(individual)
    if key not in fitness_cache:
        fitness_calls += 1
        result = evaluate_candidate(individual, trial_seed=SEED + eval_id)
        fitness_cache[key] = {
            "val_r2": result["val_r2"],
            "val_mape": result["val_mape"],
            "result": result,
        }

    return fitness_cache[key]["val_r2"]

def get_cached_metrics(individual):
    key = individual_key(individual)
    if key not in fitness_cache:
        raise KeyError("Individual not found in fitness cache; evaluate it first.")
    return {
        "val_r2": fitness_cache[key]["val_r2"],
        "val_mape": fitness_cache[key]["val_mape"],
    }

# ------------------------------------------------------------
# 11) Smoke test
# ------------------------------------------------------------
_smoke_individual = create_individual(np.random.default_rng(SEED))
_ = fitness(_smoke_individual, eval_id=1)
_smoke_metrics = get_cached_metrics(_smoke_individual)

print("\nSmoke test candidate:")
print(_smoke_individual)
print(f"Validation R²   : {_smoke_metrics['val_r2']:.6f}")
print(f"Validation MAPE : {_smoke_metrics['val_mape']:.4f}%")

# ------------------------------------------------------------
# 12) Main GA loop
# ------------------------------------------------------------
seed_everything(SEED)
generation_logs = []   # reset in case cell is rerun
ga_start = time.time()

population = initialize_population(POPULATION_SIZE, seed=SEED)
best_individual = None
best_metrics = None
all_generation_summaries = []
eval_counter = 1  # deterministic per-fitness call seeding offset

for g in range(1, GENERATIONS + 1):
    population_metrics = []
    for ind in population:
        _ = fitness(ind, eval_id=eval_counter)
        metrics = get_cached_metrics(ind)
        population_metrics.append(metrics)
        eval_counter += 1

    gen_row = log_generation_stats(generation=g, population_metrics=population_metrics)
    all_generation_summaries.append(gen_row)

    best_idx = int(np.argmax([m["val_r2"] for m in population_metrics]))
    current_best_ind = population[best_idx]
    current_best_metrics = population_metrics[best_idx]

    if (best_metrics is None) or (current_best_metrics["val_r2"] > best_metrics["val_r2"]):
        best_individual = copy.deepcopy(current_best_ind)
        best_metrics = dict(current_best_metrics)

    print(
        f"Generation {g:02d} | "
        f"best_val_R2={gen_row['best_val_r2']:.6f} | "
        f"mean_val_R2={gen_row['mean_val_r2']:.6f} | "
        f"best_val_MAPE={gen_row['best_val_mape']:.4f}% | "
        f"mean_val_MAPE={gen_row['mean_val_mape']:.4f}%"
    )

    ranked = np.argsort([-m["val_r2"] for m in population_metrics])  # descending by val_r2
    next_population = [copy.deepcopy(population[i]) for i in ranked[:ELITE_COUNT]]

    while len(next_population) < POPULATION_SIZE:
        p1 = tournament_selection(population, population_metrics, k=3, seed=SEED + g * 100 + len(next_population))
        p2 = tournament_selection(population, population_metrics, k=3, seed=SEED + g * 1000 + len(next_population))

        if np.random.rand() < CROSSOVER_RATE:
            child = crossover(p1, p2, seed=SEED + g * 10 + len(next_population))
        else:
            child = copy.deepcopy(p1)

        child = mutate(child, mutation_rate=MUTATION_RATE, seed=SEED + g * 10000 + len(next_population))
        next_population.append(child)

    population = next_population[:POPULATION_SIZE]

ga_end = time.time()
total_runtime_sec = ga_end - ga_start

# ------------------------------------------------------------
# 13) Save generation logs
# ------------------------------------------------------------
generation_logs_df = pd.DataFrame(generation_logs)
gen_logs_path = ARTIFACTS_DIR / "generation_logs.csv"
generation_logs_df.to_csv(gen_logs_path, index=False)
print("\nSaved generation logs to:", gen_logs_path.resolve())

# ------------------------------------------------------------
# 14) Convergence plots (R² + MAPE)
# ------------------------------------------------------------
plt.figure()
plt.plot(generation_logs_df["generation"], generation_logs_df["best_val_r2"], marker="o", label="Best validation R²")
plt.plot(generation_logs_df["generation"], generation_logs_df["mean_val_r2"], marker="o", label="Mean validation R²")
plt.xlabel("Generation")
plt.ylabel("Validation R² (original scale)")
plt.title("GA Convergence — Validation R²")
plt.grid(True)
plt.legend()
r2_plot_path = PLOTS_DIR / "ga_convergence_r2.png"
plt.savefig(r2_plot_path, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(generation_logs_df["generation"], generation_logs_df["best_val_mape"], marker="o", label="Best validation MAPE")
plt.plot(generation_logs_df["generation"], generation_logs_df["mean_val_mape"], marker="o", label="Mean validation MAPE")
plt.xlabel("Generation")
plt.ylabel("Validation MAPE (%)")
plt.title("GA Convergence — Validation MAPE")
plt.grid(True)
plt.legend()
mape_plot_path = PLOTS_DIR / "ga_convergence_mape.png"
plt.savefig(mape_plot_path, bbox_inches="tight")
plt.show()

print("Saved plots:")
print(" ", r2_plot_path.resolve())
print(" ", mape_plot_path.resolve())

# ------------------------------------------------------------
# 15) Final selected model (validation-only selection already done)
# ------------------------------------------------------------
final_result = evaluate_candidate(best_individual, trial_seed=SEED + 999999)
best_model = final_result["model"]

val_pred_orig = model_predict_original(best_model, X_val_tensor)
val_pred_fit  = model_predict_fit_space(best_model, X_val_tensor)
final_val_metrics = {
    **regression_metrics_original(y_val_orig, val_pred_orig),
    **regression_metrics_log(y_val_fit, val_pred_fit),
}

test_pred_orig = model_predict_original(best_model, X_test_tensor)
test_pred_fit  = model_predict_fit_space(best_model, X_test_tensor)
final_test_metrics = {
    **regression_metrics_original(y_test_orig, test_pred_orig),
    **regression_metrics_log(y_test_fit, test_pred_fit),
}

print("\n=== Final Best Individual ===")
print(best_individual)

print("\n=== Validation Metrics (Best EA Candidate) ===")
for k, v in final_val_metrics.items():
    if "mape" in k:
        print(f"{k}: {v:.4f}%")
    else:
        print(f"{k}: {v:.6f}")

print("\n=== Test Metrics (Final One-Time Evaluation) ===")
for k, v in final_test_metrics.items():
    if "mape" in k:
        print(f"{k}: {v:.4f}%")
    else:
        print(f"{k}: {v:.6f}")

# ------------------------------------------------------------
# 16) Save final artifacts
# ------------------------------------------------------------
final_summary = pd.DataFrame([
    {"split": "validation", **final_val_metrics},
    {"split": "test", **final_test_metrics},
])
final_summary_path = ARTIFACTS_DIR / "ea_final_metrics.csv"
final_summary.to_csv(final_summary_path, index=False)

best_individual_path = ARTIFACTS_DIR / "best_ea_individual.json"
with open(best_individual_path, "w", encoding="utf-8") as f:
    json.dump(best_individual, f, indent=2)

best_model_ckpt = {
    "model_state_dict": {
        k: v.detach().cpu()
        for k, v in unwrap_model(best_model).state_dict().items()
    },
    "best_individual": best_individual,
    "train_in_log_space": TRAIN_IN_LOG_SPACE,
    "clip_min_log": CLIP_MIN_LOG,
    "clip_max_log": CLIP_MAX_LOG,
    "seed": SEED,
    "use_data_parallel": USE_DATA_PARALLEL,
}
best_model_path = ARTIFACTS_DIR / "best_ea_model.pt"
torch.save(best_model_ckpt, best_model_path)

print("\nSaved final artifacts:")
print(" ", final_summary_path.resolve())
print(" ", best_individual_path.resolve())
print(" ", best_model_path.resolve())

# ------------------------------------------------------------
# 17) Compute-budget summary
# ------------------------------------------------------------
theoretical_evals = POPULATION_SIZE * GENERATIONS
actual_unique_evals = fitness_calls

print("\n=== Compute Summary ===")
print("Population size           :", POPULATION_SIZE)
print("Generations               :", GENERATIONS)
print("Theoretical evaluations   :", theoretical_evals)
print("Actual unique evaluations :", actual_unique_evals)
print("Total runtime (seconds)   :", round(total_runtime_sec, 2))
print("Total runtime (minutes)   :", round(total_runtime_sec / 60.0, 2))
print("Device used               :", device)
print("GPU count                 :", NUM_GPUS)
print("Environment snapshot      :", env_path.name)

# ------------------------------------------------------------
# 18) Completion message
# ------------------------------------------------------------
print("\nPhase 2 Step 2 COMPLETE — EA optimization executed successfully.")
print("Primary optimization metric: validation R² (original scale)")
print("Secondary tracked metric: validation MAPE (original scale)")
print("GPU-aware acceleration enabled: AMP, batched on-device inference, pinned-memory loaders.")
if USE_DATA_PARALLEL:
    print("Multi-GPU DataParallel active.")


Device: cpu
Artifacts: /Users/nadiresmail/csusb-spring/csusb_spring26_cse5140_team1/artifacts/phase2
Saved environment snapshot to: /Users/nadiresmail/csusb-spring/csusb_spring26_cse5140_team1/artifacts/phase2/environment_snapshot_phase2.json
NRP image hint: UNKNOWN

Acceleration config:
  USE_CUDA          : False
  NUM_GPUS          : 0
  USE_DATA_PARALLEL : False
  USE_AMP           : False
  NUM_WORKERS       : 4
  PIN_MEMORY        : False
  EVAL_BATCH_SIZE   : 16384

EA compute budget:
  Population size : 20
  Generations     : 15
  Theoretical eval: 300
  Mutation rate   : 0.2
  Crossover rate  : 0.8
  Elite count     : 2


/var/folders/x1/v6hpj0zn2lx7pt01f3nh0j4m0000gn/T/ipykernel_6152/4250173479.py:332: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "/Users/nadiresmail/.pyenv/versions/3.10.4/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
  File "/Users/nadiresmail/.pyenv/versions/3.10.4/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
  File "/Users/nadiresmail/.pyenv/versions/3.10.4/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
  File "/Users/nadiresmail/.pyenv/versions/3.10.4/lib/python3.10/multiprocessing/spawn.py", 

RuntimeError: DataLoader worker (pid(s) 6170, 6171, 6172, 6173) exited unexpectedly